In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!pip install -q librosa pandas numpy sentence-transformers transformers


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 48.9 MB/s eta 0:00:00


In [ ]:

import os
import csv
import gc
import numpy as np
import pandas as pd
import librosa
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from sentence_transformers import SentenceTransformer


In [ ]:

device = torch.device("cpu")

text_model = SentenceTransformer("all-MiniLM-L6-v2")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(device)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2Model were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

def extract_audio_features(audio_path):
    try:
        speech, _ = librosa.load(audio_path, sr=16000)
        inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
        with torch.no_grad():
            outputs = wav2vec_model(**inputs.to(device))
        return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    except Exception as e:
        print(f"Audio extraction error: {e}")
        return None

def extract_text_features(transcript_path):
    try:
        df = pd.read_csv(transcript_path)
        text = " ".join(df['text'].astype(str)) if 'text' in df.columns else " ".join(df.iloc[:, -1].astype(str))
        return text_model.encode(text)
    except Exception as e:
        print(f"Text extraction error: {e}")
        return None


In [ ]:

def stream_fused_features(audio_folder, transcript_folder, label_file, fused_out="fused_features.csv", label_out="labels.csv"):
    df = pd.read_csv(label_file)

    with open(fused_out, 'w', newline='') as feat_f, open(label_out, 'w', newline='') as label_f:
        feat_writer = csv.writer(feat_f)
        label_writer = csv.writer(label_f)

        for i, row in df.iterrows():
            pid = int(row['Participant_ID'])
            label = row['PHQ8_Binary']
            audio_path = os.path.join(audio_folder, f"{pid}_AUDIO.wav")
            transcript_path = os.path.join(transcript_folder, f"{pid}_TRANSCRIPT.csv")

            if os.path.exists(audio_path) and os.path.exists(transcript_path):
                audio_feat = extract_audio_features(audio_path)
                text_feat = extract_text_features(transcript_path)
                if audio_feat is not None and text_feat is not None:
                    fused = np.concatenate([audio_feat, text_feat])
                    feat_writer.writerow(fused.tolist())
                    label_writer.writerow([label])
                    print(f"✅ Saved: {pid} ({i+1}/{len(df)})")
                    del audio_feat, text_feat, fused
                    torch.cuda.empty_cache()
                    gc.collect()


In [ ]:

# 👇 CHANGE THESE PATHS
audio_dir = "/content/drive/MyDrive/audio/audio"
transcript_dir = "/content/drive/MyDrive/audio/transcript"
label_csv = "/content/drive/MyDrive/audio/label/avec_combined_labels (1).csv"

print("🔄 Starting fusion...")
stream_fused_features(audio_dir, transcript_dir, label_csv)
print("✅ Done.")


🔄 Starting fusion...
